In [8]:
import pandas as pd

In [9]:
import requests

In [10]:
df = pd.read_csv("h1b_top10_2018_2024.csv")

In [ ]:
# drop the "Total" row first and show effect
if "Country" in df.columns:
    before = len(df)
    df = df[df["Country"].str.lower() != "total"]
    after = len(df)
    print(f"dropped {before-after} total rows, remaining {after} records")
    display(df.head())

In [13]:
# remove any percentage columns from the original dataframe (case‑insensitive)
for col in list(df.columns):
    if col.lower() in {"percent", "male percent", "female percent"}:
        df = df.drop(columns=[col])
print("columns after dropping percent fields:", df.columns.tolist())

columns after dropping percent fields: ['Fiscal Year', 'Country', 'Number']


In [ ]:
# locate year columns (digits) and decide whether to melt
year_cols = [c for c in df.columns if c.isdigit()]

if len(year_cols) > 1:
    id_vars = [c for c in df.columns if c not in year_cols]
    df_long = df.melt(
        id_vars=id_vars,
        value_vars=year_cols,
        var_name="Fiscal Year",
        value_name="Number",    
    )
else:
    # data is already long-format or only contains a single year column
    df_long = df.copy()
    # if there is a column with original numbers, ensure it's named 'Number'
    if "Number_original" in df_long.columns:
        df_long = df_long.rename(columns={"Number_original": "Number"})

In [15]:

# remove unwanted percentage columns again (safety)
for col in list(df_long.columns):
    if col.lower() in {"percent", "male percent", "female percent"}:
        df_long = df_long.drop(columns=[col])

In [16]:

# keep only the year, country and number columns
keep_cols = ["Fiscal Year", "Country", "Number"]
df_long = df_long[[c for c in keep_cols if c in df_long.columns]]

In [17]:
# sort by country then year so countries appear grouped
if "Country" in df_long.columns and "Fiscal Year" in df_long.columns:
    df_long = df_long.sort_values(["Country", "Fiscal Year"]).reset_index(drop=True)

In [18]:
# keep only the selected countries
keep_countries = ["China", "India", "Korea South", "Napel", "Pakistan", "Philippines", "Taiwan"]
if "Country" in df_long.columns:
    before_filter = len(df_long)
    df_long = df_long[df_long["Country"].isin(keep_countries)]
    after_filter = len(df_long)
    print(f"filtered data to {after_filter} rows (kept {before_filter-after_filter} others)")


filtered data to 36 rows (kept 24 others)


In [19]:
# inspect result
print("final columns:", df_long.columns.tolist())
display(df_long.head(10))

final columns: ['Fiscal Year', 'Country', 'Number']


,Fiscal Year,Country,Number
12,2019,China,50609
13,2020,China,51597
14,2021,China,50328
15,2022,China,55038
16,2023,China,45344
17,2024,China,46680
18,2019,India,278491
19,2020,India,319494
20,2021,India,301616
21,2022,India,320791


In [20]:
# export cleaned data if desired
output_path = "h1b_long_cleaned.csv"
df_long.to_csv(output_path, index=False)
print(f"cleaned and sorted data saved to {output_path}")

cleaned and sorted data saved to h1b_long_cleaned.csv
